# 03 Baseline Models

Train last-value, moving-average, and linear regression baselines for each forecast horizon.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.linear_model import LinearRegression

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.evaluate import compare_model_results, regression_metrics
from src.features import get_feature_columns
from src.train_utils import time_series_train_test_split

PROCESSED_DIR = ROOT / "data" / "processed"

In [ ]:
model_df = pd.read_csv(PROCESSED_DIR / "ohio_model_ready.csv", parse_dates=["timestamp"])
train_df, test_df = time_series_train_test_split(model_df, test_size=0.2)
feature_columns = get_feature_columns(model_df)
target_columns = {
    "30min": "target_30min",
    "60min": "target_60min",
    "120min": "target_120min",
}
train_df.shape, test_df.shape

In [ ]:
results = []

for horizon, target_column in target_columns.items():
    y_true = test_df[target_column]

    last_value_pred = test_df["glucose_mgdl"]
    results.append({"model": "last_value", "horizon": horizon, **regression_metrics(y_true, last_value_pred)})

    moving_average_pred = test_df["rolling_mean_30min"]
    results.append({"model": "moving_average_30min", "horizon": horizon, **regression_metrics(y_true, moving_average_pred)})

    linear_model = LinearRegression()
    linear_model.fit(train_df[feature_columns], train_df[target_column])
    linear_pred = linear_model.predict(test_df[feature_columns])
    results.append({"model": "linear_regression", "horizon": horizon, **regression_metrics(y_true, linear_pred)})

baseline_results = compare_model_results(results)
baseline_results

In [ ]:
output_path = PROCESSED_DIR / "baseline_results.csv"
baseline_results.to_csv(output_path, index=False)
output_path